In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from datetime import datetime

CATALOG = "gwd_de"

S3_BUCKET = dbutils.secrets.get(scope="gwd_srts", key="s3_bucket_name")
RAW_BASE  = f"s3://{S3_BUCKET}/raw_data/"
metadata_file = RAW_BASE+"metadata.xlsx"
meta_df = (spark.read.format("excel").option("headerRows", 1).load(metadata_file))
display(meta_df)

In [0]:
results = []

for row in meta_df.toLocalIterator():

    source_file   = row['source_file_name']
    target_schema = row['target_schema']
    target_table  = row['target_table']

    source_path = RAW_BASE + source_file
    full_table  = f"{CATALOG}.{target_schema}.{target_table}"

    print(f"\nProcessing: {source_file} → {full_table}")

    try:
        df = (
            spark.read
            .format("excel")
            .option("headerRows", 1)
            .load(source_path)
        )

        count = df.count()
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_table)

        results.append((full_table, "SUCCESS", count))
        df.printSchema()
        

    except Exception as e:
        print(f" FAILED: {str(e)}")
        results.append((full_table, "FAILED", 0))

print("output")
for r in results:
    print(r)